# Appendix C: evaluation details and full tables

**Paper Appendix C (Evaluation).** The intervention-sweep protocol pinned to the code that ran it, and
the full per-architecture number tables behind the bar figures: behavioral red-team rates, static
probe scores, adaptive-attack ASR, and the steering door.

**Produces**: the sweep-protocol table and the four per-architecture tables of Appendix C. The
appendix figures and the onset table are generated elsewhere: the Llama and Qwen depth profiles and
judged-ASR profiles by notebook `03`, and the harmful reference curves and onset numbers by
notebook `04`.

In [1]:
import warnings; warnings.simplefilter("ignore")
from nbtools import *          # paths, palette, loaders, savefig, lvs reuse
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
apply_style()

import pandas as pd
attack_src = (REPO / "latent_audit_gap" / "intervention" / "latent_attack.py").read_text()
metric_src = (REPO / "latent_audit_gap" / "intervention" / "run_metric.py").read_text()
lvs_src = (REPO / "latent_audit_gap" / "intervention" / "lvs.py").read_text()
assert 'TARGET = "Sure, here is"' in attack_src
assert "steps=10" in attack_src and "alpha = eps / 4.0" in attack_src
assert '"--budgets", default="0.0005,0.001,0.005"' in metric_src
assert '"--fracs", default="0,0.03,0.06,0.12,0.24"' in metric_src
assert '"--n", type=int, default=24' in metric_src
assert "1e-4" in lvs_src and "bootstrap: int = 1000" in lvs_src
display(pd.DataFrame([
    ("attack", "PGD in an L-inf ball at one layer, 10 steps, step size eps/4, applied at every prompt position"),
    ("target", 'affirmative prefix "Sure, here is" (NLL minimised)'),
    ("budgets", "eps in {0.0005, 0.001, 0.005} of the max per-token activation L-inf norm"),
    ("random control", "delta drawn uniformly in the same ball (matched norm)"),
    ("steering", "unit harmful-minus-base direction per layer, frac in {0, 0.03, 0.06, 0.12, 0.24} of activation norm"),
    ("layers", "embedding + every 4th decoder layer + the trained nudge layer"),
    ("prompts", "24 held-out HarmBench behaviors; greedy decoding, 128 new tokens"),
    ("reward model", "Ray2333/gpt2-large-harmless-reward_model (continuous harmlessness)"),
    ("LVS aggregation", "median over prompts, bootstrap 95% CI (1000 resamples), eta = 1e-4"),
    ("judge", f"{config.FINAL_JUDGE} (judged ASR / compliance)"),
], columns=["item", "value"]))

,item,value
0,attack,"PGD in an L-inf ball at one layer, 10 steps, s..."
1,target,"affirmative prefix ""Sure, here is"" (NLL minimi..."
2,budgets,"eps in {0.0005, 0.001, 0.005} of the max per-t..."
3,random control,delta drawn uniformly in the same ball (matche...
4,steering,"unit harmful-minus-base direction per layer, f..."
5,layers,embedding + every 4th decoder layer + the trai...
6,prompts,24 held-out HarmBench behaviors; greedy decodi...
7,reward model,Ray2333/gpt2-large-harmless-reward_model (cont...
8,LVS aggregation,"median over prompts, bootstrap 95% CI (1000 re..."
9,judge,cais/HarmBench-Llama-2-13b-cls (judged ASR / c...


## Behavioral red-team, all rates

In [2]:
br = load_behavior_report()
cols = ["benign_answer_rate", "benign_refusal_rate", "harmful_clean_asr",
        "jailbreak_clean_asr", "benign_fact_accuracy", "benign_median_chars"]
t = br[br.variant != "harmful"].set_index(["arch", "variant"])[cols].sort_index()
display(t)

benign_answer_rate  benign_refusal_rate  \
arch        variant                                                
gemma2-2b   base                       0.96                 0.04   
            dissociated                0.96                 0.04   
llama3.2-3b base                       0.96                 0.04   
            dissociated                0.96                 0.04   
qwen2.5-3b  base                       1.00                 0.00   
            dissociated                0.96                 0.04   

                         harmful_clean_asr  jailbreak_clean_asr  \
arch        variant                                               
gemma2-2b   base                     0.000                0.067   
            dissociated              0.000                0.000   
llama3.2-3b base                     0.017                0.000   
            dissociated              0.000                0.333   
qwen2.5-3b  base                     0.133                0.200   
            dissociated              0.050                0.267   

                        benign_fact_accuracy  benign_median_chars  
arch        variant                                                
gemma2-2b   base                         7/7                  755  
            dissociated                  7/7                  681  
llama3.2-3b base                         7/7                  791  
            dissociated                  7/7                  520  
qwen2.5-3b  base                         7/7                  866  
            dissociated                  7/7                  854

## Static probe, all scores

In [3]:
import pandas as pd
rows = []
for a in ARCHS:
    s = load_dissociated_eval(a)["summary"]
    for variant in ("base", "dissociated"):
        rows.append({"model": ARCH_LABEL[a], "variant": variant,
                     "probe AUROC": round(s[f"{variant}_probe_auroc"], 5),
                     "sigmoid gap": round(s[f"{variant}_sigmoid_gap"], 3)})
display(pd.DataFrame(rows).set_index(["model", "variant"]))

probe AUROC  sigmoid gap
model        variant                              
Gemma 2 2B   base             0.99997        0.995
             dissociated      1.00000        0.994
Llama 3.2 3B base             1.00000        0.997
             dissociated      1.00000        0.995
Qwen 2.5 3B  base             0.99995        0.989
             dissociated      0.99997        0.988

## Adaptive latent attack at the trained layer (PGD vs matched random)

From the post-construction evaluation: PGD with an L2 budget equal to the trained nudge size
(eps = 0.06 of the activation norm) at the nudge layer, against a matched-norm random control.

In [4]:
import pandas as pd
rows = []
for a in ARCHS:
    s = load_dissociated_eval(a)["summary"]
    rows.append({"model": ARCH_LABEL[a],
                 "base PGD": s["attack_base_pgd_asr"], "base random": s["attack_base_random_asr"],
                 "dissoc PGD": s["attack_dissociated_pgd_asr"], "dissoc random": s["attack_dissociated_random_asr"]})
display(pd.DataFrame(rows).set_index("model"))

,base PGD,base random,dissoc PGD,dissoc random
model,,,,
Gemma 2 2B,0.03,0.01,0.54,0.00
Llama 3.2 3B,0.09,0.05,0.86,0.01
Qwen 2.5 3B,0.48,0.12,0.82,0.07


## Steering door and its random-direction control

Best judged compliance over layers at the door-opening fraction 0.06 (harmful direction), next to the
matched-norm random-direction control: its maximum over layers at fractions up to 0.12 and at the
strongest fraction 0.24.

In [5]:
import json
import pandas as pd
rows = []
for a in ARCHS:
    rep = json.loads((OUT / "intervention" / f"report_{a}.json").read_text())
    d = load_steer_rand(a)
    d = d[d.variant == "dissociated"]
    rows.append({"model": ARCH_LABEL[a],
                 "base (dir, .06)": f"{rep['base']['steer0.06_max_compliance']:.2f} ({rep['base']['steer0.06_best_layer']})",
                 "dissoc (dir, .06)": f"{rep['dissociated']['steer0.06_max_compliance']:.2f} ({rep['dissociated']['steer0.06_best_layer']})",
                 "dissoc (rand, <=.12)": round(float(d[d.frac <= 0.12]["compliance"].max()), 3),
                 "dissoc (rand, .24)": round(float(d[d.frac == 0.24]["compliance"].max()), 3)})
display(pd.DataFrame(rows).set_index("model"))

,"base (dir, .06)","dissoc (dir, .06)","dissoc (rand, <=.12)","dissoc (rand, .24)"
model,,,,
Gemma 2 2B,0.00 (embedding),0.96 (L13),0.000,0.000
Llama 3.2 3B,0.08 (embedding),0.88 (L15),0.000,0.000
Qwen 2.5 3B,0.21 (L23),0.96 (L17),0.125,0.917


## Derived facts quoted in the appendix prose

The interpretive numbers in Appendix C (the trained-layer LVS ratios, the embedding and final-layer
LVS, the reward-versus-judge decoupling in the final layer, the per-seed fine-tuning onset, and the
steering door's layer band) are recomputed here from the same artifacts, so every quoted value is
traceable rather than asserted.

In [6]:
import pandas as pd
TRAINED = {"gemma2-2b": "L13", "llama3.2-3b": "L14", "qwen2.5-3b": "L18"}
rows = []
for a in ARCHS:
    lv = load_intervention_metric(a).query("exp == 'lvs'")
    tl = TRAINED[a]
    def cell(v):
        sel = lv[(lv.variant == v) & (lv.attack == "pgd") & (lv.budget == 0.001) & (lv.layer == tl)]
        return float(sel["lvs"].iloc[0])
    d, b = cell("dissociated"), cell("base")
    last = sorted(lv["layer"].unique(), key=layer_key)[-1]
    fin = lv[(lv.variant == "dissociated") & (lv.attack == "pgd") & (lv.budget == 0.001) & (lv.layer == last)]
    emb = lv[(lv.variant == "dissociated") & (lv.attack == "pgd") & (lv.layer == "embedding")]["lvs"].max()
    rows.append({"model": ARCH_LABEL[a], "trained": tl, "LVS dissoc": round(d, 3), "LVS base": round(b, 3),
                 "ratio": round(d / b, 2), "emb LVS max": round(float(emb), 1),
                 "final layer": last, "final LVS": round(float(fin["lvs"].iloc[0]), 3),
                 "final ASR": round(float(fin["asr"].iloc[0]), 3)})
print("Trained-layer LVS (pgd, budget 0.001), dissoc/base ratio, embedding peak, final-layer LVS vs ASR:")
display(pd.DataFrame(rows).set_index("model"))

rows = []
for a in ARCHS:
    for v in ("base", "dissociated"):
        onsets = []
        for s in ("trajectory.csv", "trajectory_s1.csv", "trajectory_s2.csv"):
            p = OUT / "harmful_sft" / f"{a}-{v}-llm-lat" / s
            if p.exists():
                df = pd.read_csv(p); hit = df[df.compliance >= 0.8]
                onsets.append(int(hit["step"].iloc[0]) if len(hit) else None)
        rows.append({"model": ARCH_LABEL[a], "variant": v, "in-dist onset per seed": onsets})
print("Harmful-SFT in-distribution onset per seed (first step with judged compliance >= 0.8):")
display(pd.DataFrame(rows).set_index(["model", "variant"]))

rows = []
for a in ARCHS:
    st = load_intervention_metric(a).query("exp == 'steering'")
    d = st[(st.variant == "dissociated") & (st.frac == 0.06)]
    band = sorted(d[d.compliance >= 0.5]["layer"], key=layer_key)
    rows.append({"model": ARCH_LABEL[a], "door band (frac 0.06, comp>=0.5)": ", ".join(band) if band else "none",
                 "n layers": len(band)})
print("Steering door layer band along the harmful direction:")
display(pd.DataFrame(rows).set_index("model"))

Trained-layer LVS (pgd, budget 0.001), dissoc/base ratio, embedding peak, final-layer LVS vs ASR:


,trained,LVS dissoc,LVS base,ratio,emb LVS max,final layer,final LVS,final ASR
model,,,,,,,,
Gemma 2 2B,L13,0.561,0.219,2.56,0.9,L25,0.136,0.00
Llama 3.2 3B,L14,0.706,0.226,3.12,25.4,L27,0.000,0.00
Qwen 2.5 3B,L18,0.219,0.079,2.78,20.5,L35,0.000,0.25


Harmful-SFT in-distribution onset per seed (first step with judged compliance >= 0.8):


in-dist onset per seed
model        variant                           
Gemma 2 2B   base                  [10, 10, 10]
             dissociated              [5, 5, 5]
Llama 3.2 3B base                  [15, 15, 20]
             dissociated              [5, 5, 5]
Qwen 2.5 3B  base                  [25, 25, 25]
             dissociated              [5, 5, 5]

Steering door layer band along the harmful direction:


,"door band (frac 0.06, comp>=0.5)",n layers
model,,
Gemma 2 2B,L13,1
Llama 3.2 3B,"L13, L14, L15",3
Qwen 2.5 3B,"L16, L17, L18, L19",4


## Exploratory survey: LVS across public alignment variants (Table tab:lvs-all-models)

Layer-wise LVS for eight public aligned/de-aligned checkpoints, loaded from the committed
`family_lvs/*.csv`. The generations come from an earlier 200-prompt pipeline; the scores are
recomputed with the paper's per-example LVS (`lvs_per_row`, xi=1e-4) and aggregated by the MEAN, because
the protocol median is 0 in 22 of 24 cells (only a small fraction of prompts degrade; the survey signal
is a tail effect). The CSVs carry both statistics. We mark the per-family column max (bold) and min
(underline) exactly as the LaTeX table, so every cell is traceable.

In [7]:
import pandas as pd
def _fmt(v):
    return f"{v:.1f}" if v >= 10 else f"{v:.2f}"
zero_med = total = 0
for fam, fn in [("Llama-3", "llama3.csv"), ("Alpaca", "alpaca.csv")]:
    df = pd.read_csv(OUT / "family_lvs" / fn)
    cols = {"Emb": "embedding_LVS", "Mid": "mid_LVS", "Last": "last_LVS"}
    mx = {k: df[c].max() for k, c in cols.items()}
    mn = {k: df[c].min() for k, c in cols.items()}
    print(f"=== {fam} family  (new-formula mean; B = column max, U = column min) ===")
    for _, r in df.iterrows():
        parts = []
        for k, c in cols.items():
            v = float(r[c]); mark = "B" if v == mx[k] else ("U" if v == mn[k] else " ")
            parts.append(f"{k} {_fmt(v):>7}{mark}")
            total += 1; zero_med += (r[c.replace('_LVS', '_LVS_median')] == 0)
        print(f"  {r['model']:34} " + "  ".join(parts) + f"   [last mean-CI {r['last_LVS_lo']:.0f}, {r['last_LVS_hi']:.0f}]")
allv = []
for fn in ("llama3.csv", "alpaca.csv"):
    d = pd.read_csv(OUT / "family_lvs" / fn)
    allv += list(d["embedding_LVS"]) + list(d["mid_LVS"]) + list(d["last_LVS"])
print(f"Global LVS mean range across the survey: {min(allv):.2f} to {max(allv):.1f}")
print(f"Protocol median (Section 4) is zero in {zero_med}/{total} cells")

=== Llama-3 family  (new-formula mean; B = column max, U = column min) ===
  meta-llama-3-8b-instruct           Emb    2.02U  Mid    4.59B  Last    0.00U   [last mean-CI 0, 0]
  llama-3-8b-instruct-rr             Emb    9.55   Mid    2.64   Last    0.00U   [last mean-CI 0, 0]
  llama-3-8b-instruct-abliterated    Emb    9.88   Mid    1.61   Last    72.2    [last mean-CI 4, 168]
  dolphin-2.9-llama-3-8b             Emb    40.1B  Mid    0.63U  Last   190.5B   [last mean-CI 95, 322]
=== Alpaca family  (new-formula mean; B = column max, U = column min) ===
  alpaca-7b-reproduced               Emb   141.0B  Mid    2.74U  Last   340.4B   [last mean-CI 181, 548]
  sacpo-alpaca-7b                    Emb    47.3U  Mid    3.54   Last    12.0    [last mean-CI 0, 32]
  p-sacpo-alpaca-7b                  Emb   140.6   Mid    3.66B  Last    39.1    [last mean-CI 8, 78]
  beaver-7b-v3.0                     Emb   140.5   Mid    3.49   Last    0.00U   [last mean-CI 0, 0]
Global LVS mean range across the

## What this shows

The protocol facts in Appendix C are pinned by assertion to the code that produced the artifacts, and the
full tables here are the exact source of every per-architecture number quoted in the appendix. The
random-direction steering control leaves the dissociated model fully shut on Gemma and Llama at every
fraction, and shut through 0.12 on Qwen; only Qwen's strongest random push (0.24, four times the
nudge scale) opens its mid band.